### Dataset

In [1]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny"
)

c:\Users\Federico\DesktopW\patchAliasing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import pandas as pd
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-tiny")

# Load historical data
db_url = "dataset/Dataset-SolarTechLabEngineered.csv"
df = pd.read_csv(db_url, delimiter=",")

In [8]:
df.columns

Index(['Date', 'Power_W', 'Temp_Celsius', 'Irradiance_horizontalPlane',
       'Irradiance_inclinedPlane', 'Wind_speed', 'Wind_direction'],
      dtype='str')

In [13]:
df['item_id'] = 'power_series'  # Add an item_id column for the time series identifier


df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["item_id", "Date"]).reset_index(drop=True)

pred_df = pipeline.predict_df(
    df,
    prediction_length=64,
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id",
    timestamp_column="Date",
    target="Power_W",
    freq="min",
)

ValueError: Could not infer frequency for series power_series

In [ ]:
pred_df.head()

,item_id,Month,target_name,predictions,0.1,0.5,0.9
0,T1,1961-01-01,#Passengers,442.025238,418.598541,443.874756,472.644379
1,T1,1961-02-01,#Passengers,445.313141,414.899384,447.984711,468.534424
2,T1,1961-03-01,#Passengers,462.472229,429.489807,462.369507,492.166504
3,T1,1961-04-01,#Passengers,479.733978,445.724121,478.809296,507.989777
4,T1,1961-05-01,#Passengers,515.387817,458.259430,518.881226,560.391724


In [ ]:
from chronos import BaseChronosPipeline
import torch

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cpu",
)

context = torch.tensor([1.2, 1.5, 1.7, 1.4, 1.8])

token_ids, attention_mask, tokenizer_state = pipeline.tokenizer.context_input_transform(context.unsqueeze(0))


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)